# Inferencia

**Proyecto:** House Pricing — House Prices: Advanced Regression Techniques (Kaggle)

**Notebook:** `4.0-ae-inferencia.ipynb` (convención Cookiecutter Data Science)

Este notebook genera las predicciones de `SalePrice` para el set de test de Kaggle. Usa como entrada los
**datos crudos** de `data/raw/test.csv` y el pipeline resultado `models/best_model.joblib` (creado en el
notebook 3.0), que incluye la selección de features, la ingeniería de características completa, el
modelo ganador y la reversión de `log1p` a dólares. No se aplica ninguna transformación manual.

La salida es `data/processed/predictions.csv` con el formato de envío de Kaggle (`Id`, `SalePrice`).

## 0. Librerías y configuración

Importa las librerías y agrega la raíz del proyecto al path para usar las rutas de `house_pricing.config`.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from house_pricing.config import MODELS_DIR, PROCESSED_DATA_DIR, RAW_DATA_DIR

2026-11-09 21:21:44.954 | INFO     | house_pricing.config:<module>:11 - PROJ_ROOT path is: C:\Users\Anderson Elias\OneDrive - Universidad Galileo\8vo. Trimestre\Product Development - B\house_pricing


## 1. Carga del pipeline resultado

Carga `models/best_model.joblib` y muestra su estructura: `TransformedTargetRegressor` →
`Pipeline(select → fe → model)`.

In [2]:
best_model = joblib.load(MODELS_DIR / "best_model.joblib")

model_step = best_model.regressor_.named_steps["model"]
print(f"Modelo: {type(model_step).__name__} | {model_step.get_params()}")
best_model

Modelo: ElasticNet | {'alpha': 0.0005, 'copy_X': True, 'fit_intercept': True, 'l1_ratio': 0.2, 'max_iter': 50000, 'positive': False, 'precompute': False, 'random_state': None, 'selection': 'cyclic', 'tol': 0.0001, 'warm_start': False}


,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...iter=50000))])
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](80,)","['Id','MSSubClass','MSZoning',...,'YrSold','SaleType','SaleCondition']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,80
regressor_ regressor_: objectFitted regressor.,Pipeline,Pipeline(step...iter=50000))])
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('select', ...), ('fe', ...), ...]"


## 2. Carga de los datos crudos de test

Lee `data/raw/test.csv` sin ninguna modificación: el pipeline se encarga de seleccionar las features,
imputar, codificar, tratar outliers, transformar y escalar.

In [3]:
test_raw = pd.read_csv(RAW_DATA_DIR / "test.csv")
print(f"test: {test_raw.shape}")
test_raw.head()

test: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


## 3. Predicción

Aplica el pipeline resultado a los datos crudos. Las predicciones ya salen en dólares porque el
`TransformedTargetRegressor` revierte el `log1p` con `expm1`.

In [4]:
predictions = best_model.predict(test_raw)

submission = pd.DataFrame({"Id": test_raw["Id"], "SalePrice": predictions})
submission["SalePrice"].describe().to_frame().style.format("{:,.0f}")

,SalePrice
count,"1,459"
mean,"178,062"
std,"75,893"
min,"55,254"
25%,"124,799"
50%,"158,268"
75%,"212,311"
max,"517,305"


## 4. Validación y guardado de las predicciones

Verifica que el archivo cumpla el formato de Kaggle comparándolo con `sample_submission.csv` (mismas
columnas, mismos `Id` en el mismo orden), que no haya valores faltantes y que todos los precios sean
positivos. Después lo guarda en `data/processed/predictions.csv`.

In [5]:
sample = pd.read_csv(RAW_DATA_DIR / "sample_submission.csv")

assert list(submission.columns) == list(sample.columns), "Columnas distintas a sample_submission.csv"
assert submission["Id"].equals(sample["Id"]), "Los Id no coinciden con sample_submission.csv"
assert submission["SalePrice"].notna().all(), "Hay predicciones faltantes"
assert (submission["SalePrice"] > 0).all(), "Hay precios no positivos"

submission.to_csv(PROCESSED_DATA_DIR / "predictions.csv", index=False)
print(f"Guardado en {PROCESSED_DATA_DIR / 'predictions.csv'} ({len(submission)} filas)")
submission.head()

Guardado en C:\Users\Anderson Elias\OneDrive - Universidad Galileo\8vo. Trimestre\Product Development - B\house_pricing\data\processed\predictions.csv (1459 filas)


,Id,SalePrice
0,1461,119905.846812
1,1462,157363.893525
2,1463,176114.575078
3,1464,190688.778139
4,1465,216315.283031
